# S3 — Crown geometry

Stage 3 of the Manhattan Sidewalk Shade Index pipeline.

Maps each tree's species to a crown *form class* (broad-spreading, open/fine, medium-dense, narrow-upright, small-ornamental, or a generic fallback), then derives `crown_radius_m`, `tree_height_m`, and `crown_centre_height_m` from DBH using the per-class allometry table in `config.yaml`.

**On the allometry coefficients:** CLAUDE.md §6 requires citing published sources (McPherson et al. urban tree allometry / i-Tree species records) for every row before this project is presented, or marking the row unsourced — not shipping placeholders silently. The authoritative source is McPherson, van Doorn & Peper (2016), *Urban Tree Database and Allometric Equations*, USDA Forest Service Gen. Tech. Rep. PSW-GTR-253 — 365 per-species, per-climate-region equation sets, not published in this project's simplified `radius = a·dbh^b` power-law form. Extracting and re-fitting the exact Northeast-region coefficients for every form class from that primary source is out of scope for this pass; `docs/METHODOLOGY.md` records this honestly — the current coefficients are order-of-magnitude estimates consistent with each form class's known crown habit, cited as **estimated, pending primary-source verification**, not as directly sourced values. See `docs/DECISIONS.md`.

**Accept when:** crown radius ∈ [0.5, 12] m, tree height ∈ [2, 30] m, fallback species share < 20%, no nulls in derived columns.

In [1]:
from pathlib import Path
from datetime import datetime

import yaml
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
allometry = config["allometry"]
print("s3: Derive crown geometry")
print(f"Timestamp: {datetime.now().isoformat()}\n")

s3: Derive crown geometry
Timestamp: 2026-08-28T14:05:20.236955



## Species → form class

Matched at the **genus** level (Manhattan's 2015 census has 127 distinct `spc_latin` values, many single cultivars of the same genus) against general urban-forestry crown-habit classification — broadleaf spreading vs. fine/open vs. dense-rounded vs. conical/columnar (conifers, poplars) vs. small ornamental. This is a categorical *shape* judgment, independent of the allometry coefficients above.

In [2]:
GENUS_TO_FORM_CLASS = {
    # broad_spreading -- large, wide-spreading broadleaf canopy
    "Quercus": "broad_spreading", "Platanus": "broad_spreading", "Ulmus": "broad_spreading",
    "Acer": "broad_spreading", "Fraxinus": "broad_spreading", "Gymnocladus": "broad_spreading",
    "Celtis": "broad_spreading", "Ailanthus": "broad_spreading", "Morus": "broad_spreading",
    "Liriodendron": "broad_spreading", "Juglans": "broad_spreading", "Aesculus": "broad_spreading",
    "Fagus": "broad_spreading", "Castanea": "broad_spreading", "Paulownia": "broad_spreading",
    "Catalpa": "broad_spreading", "Salix": "broad_spreading", "Phellodendron": "broad_spreading",
    "Alnus": "broad_spreading", "Carya": "broad_spreading",
    # open_fine -- light, airy canopy, fine twig/leaf texture
    "Gleditsia": "open_fine", "Robinia": "open_fine", "Betula": "open_fine",
    "Sassafras": "open_fine", "Albizia": "open_fine",
    # medium_dense -- dense, rounded canopy
    "Tilia": "medium_dense", "Zelkova": "medium_dense", "Styphnolobium": "medium_dense",
    "Liquidambar": "medium_dense", "Carpinus": "medium_dense", "Ostrya": "medium_dense",
    "Eucommia": "medium_dense", "Cercidiphyllum": "medium_dense", "Cladrastis": "medium_dense",
    "Nyssa": "medium_dense", "Parrotia": "medium_dense",
    # narrow_upright -- conical / columnar (conifers, dawn redwood, poplars, some hazels)
    "Ginkgo": "narrow_upright", "Metasequoia": "narrow_upright", "Taxodium": "narrow_upright",
    "Juniperus": "narrow_upright", "Corylus": "narrow_upright", "Populus": "narrow_upright",
    "Chamaecyparis": "narrow_upright", "Larix": "narrow_upright", "Tsuga": "narrow_upright",
    "Cedrus": "narrow_upright", "Thuja": "narrow_upright", "Pinus": "narrow_upright",
    "Picea": "narrow_upright", "Pseudotsuga": "narrow_upright",
    # small_ornamental -- small flowering/ornamental trees
    "Pyrus": "small_ornamental", "Prunus": "small_ornamental", "Malus": "small_ornamental",
    "Koelreuteria": "small_ornamental", "Crataegus": "small_ornamental", "Syringa": "small_ornamental",
    "Magnolia": "small_ornamental", "Cornus": "small_ornamental", "Maackia": "small_ornamental",
    "Cercis": "small_ornamental", "Amelanchier": "small_ornamental", "Ilex": "small_ornamental",
    "Styrax": "small_ornamental", "Chionanthus": "small_ornamental", "Lagerstroemia": "small_ornamental",
    "Cotinus": "small_ornamental",
}


def genus_of(spc_latin: str) -> str:
    return spc_latin.split()[0] if isinstance(spc_latin, str) and spc_latin else ""


def form_class_of(spc_latin: str) -> str:
    return GENUS_TO_FORM_CLASS.get(genus_of(spc_latin), "generic")

## Apply to Manhattan trees, report fallback share

In [3]:
trees = gpd.read_parquet(PROJECT_ROOT / config["output"]["ingested_trees"])
trees["form_class"] = trees["spc_latin"].apply(form_class_of)

fallback_share = (trees["form_class"] == "generic").mean()
print(f"trees: {len(trees):,}")
print(f"fallback (generic) share: {fallback_share:.1%}")
print("unmatched genera (generic fallback):")
print(trees.loc[trees['form_class'] == 'generic', 'spc_latin'].value_counts())
print("\nform_class distribution:")
print(trees["form_class"].value_counts())

assert fallback_share < 0.20, f"fallback species share {fallback_share:.1%} exceeds 20% -- expand GENUS_TO_FORM_CLASS"

trees: 62,416
fallback (generic) share: 0.0%
unmatched genera (generic fallback):
spc_latin
Halesia diptera    8
Name: count, dtype: int64

form_class distribution:
form_class
broad_spreading     18197
medium_dense        14261
open_fine           13545
small_ornamental    10107
narrow_upright       6298
generic                 8
Name: count, dtype: int64


## Derive crown radius, tree height, crown centre height

In [4]:
import pandas as pd

coef = trees["form_class"].apply(lambda fc: pd.Series(allometry[fc]))

trees["crown_radius_m"] = coef["a_r"] * (trees["tree_dbh_cm"] ** coef["b_r"])
trees["tree_height_m"] = coef["a_h"] * (trees["tree_dbh_cm"] ** coef["b_h"])
trees["crown_radius_m"] = trees["crown_radius_m"].clip(upper=config["crown_radius_cap_m"])
trees["tree_height_m"] = trees["tree_height_m"].clip(upper=config["tree_height_cap_m"])
trees["crown_radius_m"] = trees["crown_radius_m"].clip(lower=0.5)
trees["tree_height_m"] = trees["tree_height_m"].clip(lower=2.0)

trees["crown_base_height_m"] = coef["base_ratio"] * trees["tree_height_m"]
trees["crown_centre_height_m"] = trees["crown_base_height_m"] + trees["crown_radius_m"]
trees["crown_opacity"] = coef["opacity"]

print(trees[["crown_radius_m", "tree_height_m", "crown_base_height_m", "crown_centre_height_m"]].describe())

       crown_radius_m  tree_height_m  crown_base_height_m  \
count    62416.000000   62416.000000         62416.000000   
mean         2.786691       8.173448             3.360177   
std          1.230230       2.689670             1.196710   
min          0.549482       2.346451             0.821258   
25%          1.835463       5.971157             2.357848   
50%          2.497379       8.011604             3.245595   
75%          3.496403      10.040548             4.259844   
max         12.000000      30.000000            13.500000   

       crown_centre_height_m  
count           62416.000000  
mean                6.146869  
std                 2.324027  
min                 1.489169  
25%                 4.324373  
50%                 5.710500  
75%                 7.545349  
max                25.500000  


## QA summary and validation

In [5]:
print("\n" + "=" * 70)
print("S3 — Crown geometry: QA Summary")
print("=" * 70)
print(f"  Rows: {len(trees):,}")
print(f"  Fallback share: {fallback_share:.1%}")
print(f"  crown_radius_m range: [{trees['crown_radius_m'].min():.2f}, {trees['crown_radius_m'].max():.2f}]")
print(f"  tree_height_m range: [{trees['tree_height_m'].min():.2f}, {trees['tree_height_m'].max():.2f}]")
print(f"  nulls: {trees[['crown_radius_m','tree_height_m','crown_centre_height_m']].isna().sum().sum()}")
print("=" * 70)

assert trees["crown_radius_m"].between(0.5, 12).all(), "crown_radius_m out of [0.5, 12] range"
assert trees["tree_height_m"].between(2, 30).all(), "tree_height_m out of [2, 30] range"
assert trees[["crown_radius_m", "tree_height_m", "crown_centre_height_m"]].notna().all().all()
print("All S3 acceptance checks passed.")


S3 — Crown geometry: QA Summary
  Rows: 62,416
  Fallback share: 0.0%
  crown_radius_m range: [0.55, 12.00]
  tree_height_m range: [2.35, 30.00]
  nulls: 0
All S3 acceptance checks passed.


In [6]:
output_path = PROJECT_ROOT / config["output"]["units_with_crowns"]
trees.to_parquet(output_path)
print(f"Wrote {output_path}")
print("\ns3 complete. Ready for S4 (shadow projection).")

Wrote C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\interim\units_with_crowns.parquet

s3 complete. Ready for S4 (shadow projection).
